# 🚀 Full Evaluation Run
**Workflows Over Weights: Agentic Scaling for Small Models on HLE-Verified**

This notebook runs the full 641-question evaluation across all 4 ablation stages.

**Session plan:**
- ~75 questions per Colab session (~240 min)
- ~9 sessions total (641 / 75)
- Checkpoint saved to Drive after **every question** — fully resumable

**Just run all cells.** The notebook automatically resumes from the last checkpoint.

---
## 1. Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
%%capture
%pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    bitsandbytes>=0.43.0 \
    torch \
    tavily-python \
    duckduckgo-search \
    pydantic \
    python-dotenv

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import os, sys, json, time, logging, re

DRIVE_DIR = '/content/drive/MyDrive/workflows-over-weights'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Clone / pull repo
REPO_URL = 'https://github.com/ajanm007/workflows-over-weights.git'  # UPDATE with your repo URL
REPO_DIR = '/content/workflows-over-weights'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

---
## 2. Configuration

In [ ]:
# ============================================================
# SESSION CONFIG — adjust per session if needed
# ============================================================
QUESTIONS_PER_SESSION = 75     # Colab T4 budget: ~240 min, ~3 min/question
CHECKPOINT_FILE = os.path.join(DRIVE_DIR, 'full_run_results.json')
RUN_LOG_FILE = os.path.join(DRIVE_DIR, 'run_log.jsonl')

print(f'Checkpoint: {CHECKPOINT_FILE}')
print(f'Run log: {RUN_LOG_FILE}')
print(f'Questions per session: {QUESTIONS_PER_SESSION}')

---
## 3. Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from config import MAX_TOKENS

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
print(f'Model loaded. GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB / {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')

---
## 4. Inference & Pipeline Functions

In [ ]:
from pipeline.prompts import BASE_PROMPT, TOOL_AUGMENTED_PROMPT, SELF_CRITIQUE_PROMPT_SYSTEM, SELF_CRITIQUE_PROMPT_USER
from tools.web_search import web_search
from eval.scorer import score_exact_match

def generate_response(prompt: str, max_new_tokens: int = MAX_TOKENS, temperature: float = 0.1) -> str:
    """Generate a response from the Qwen model."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

def extract_search_query(text: str) -> str | None:
    match = re.search(r'<SEARCH>(.*?)</SEARCH>', text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

def extract_final_answer(text: str) -> str | None:
    match = re.search(r'<FINAL_ANSWER>(.*?)</FINAL_ANSWER>', text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

def run_stage_1(item):
    prompt = BASE_PROMPT.format(question=item['question'])
    response = generate_response(prompt)
    prediction = extract_final_answer(response) or response
    return {'response': response, 'prediction': prediction}

def run_stage_2(item):
    prompt = TOOL_AUGMENTED_PROMPT.format(question=item['question'])
    response = generate_response(prompt)
    
    query = extract_search_query(response)
    search_provider = None
    if query:
        search_results = web_search(query)
        search_provider = 'tavily' if 'Tavily' in search_results else 'duckduckgo'
        prompt += f'\n\nSearch results for "{query}":\n{search_results}\n\nBased on the search results, provide your final answer with <FINAL_ANSWER>your answer</FINAL_ANSWER>.'
        response = generate_response(prompt)
    
    prediction = extract_final_answer(response) or response
    return {'response': response, 'prediction': prediction, 'search_query': query, 'search_provider': search_provider}

def run_stage_3(item, s1_response):
    combined = f'{SELF_CRITIQUE_PROMPT_SYSTEM}\n{SELF_CRITIQUE_PROMPT_USER.format(question=item["question"], candidate_response=s1_response)}'
    response = generate_response(combined)
    prediction = extract_final_answer(response) or response
    return {'response': response, 'prediction': prediction}

def run_stage_4(item, s2_response):
    combined = f'{SELF_CRITIQUE_PROMPT_SYSTEM}\n{SELF_CRITIQUE_PROMPT_USER.format(question=item["question"], candidate_response=s2_response)}'
    response = generate_response(combined)
    prediction = extract_final_answer(response) or response
    return {'response': response, 'prediction': prediction}

print('Pipeline functions loaded.')

---
## 5. Load Dataset & Checkpoint

In [ ]:
from config import DATASET_PATH

# Load dataset
if DATASET_PATH.exists():
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
else:
    drive_dataset = os.path.join(DRIVE_DIR, 'hle_verified_gold.json')
    with open(drive_dataset, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

# Load checkpoint
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        results = json.load(f)
else:
    results = {}

total_items = len(dataset)
done = len(results)
remaining = total_items - done
this_session = min(QUESTIONS_PER_SESSION, remaining)

print(f'Dataset: {total_items} items')
print(f'Already completed: {done}')
print(f'Remaining: {remaining}')
print(f'This session target: {this_session}')
print(f'Estimated sessions left: {max(1, remaining // QUESTIONS_PER_SESSION)}')

---
## 6. Run Inference Loop

In [ ]:
session_start = time.time()
session_count = 0
session_correct = {s: 0 for s in ['1', '2', '3', '4']}
errors = []

for idx, item in enumerate(dataset):
    qid = str(item.get('id', idx))
    
    # Skip already completed
    if qid in results:
        continue
    
    # Check session budget
    if session_count >= QUESTIONS_PER_SESSION:
        print(f'\nSession budget reached ({QUESTIONS_PER_SESSION} questions). Stopping.')
        break
    
    session_count += 1
    overall_progress = len(results) + 1
    
    print(f'\n[{session_count}/{this_session}] (overall: {overall_progress}/{total_items}) QID: {qid}')
    
    expected_ans = item.get('answer', '')
    ans_type = item.get('answer_type', 'exactMatch')
    
    item_results = {'expected': expected_ans, 'type': ans_type, 'stages': {}}
    t0 = time.time()
    
    try:
        # Stage 1
        s1 = run_stage_1(item)
        s1['correct'] = score_exact_match(s1['prediction'], expected_ans, ans_type)
        item_results['stages']['1'] = s1
        if s1['correct']: session_correct['1'] += 1
        
        # Stage 2
        s2 = run_stage_2(item)
        s2['correct'] = score_exact_match(s2['prediction'], expected_ans, ans_type)
        item_results['stages']['2'] = s2
        if s2['correct']: session_correct['2'] += 1
        
        # Stage 3
        s3 = run_stage_3(item, s1['response'])
        s3['correct'] = score_exact_match(s3['prediction'], expected_ans, ans_type)
        item_results['stages']['3'] = s3
        if s3['correct']: session_correct['3'] += 1
        
        # Stage 4
        s4 = run_stage_4(item, s2['response'])
        s4['correct'] = score_exact_match(s4['prediction'], expected_ans, ans_type)
        item_results['stages']['4'] = s4
        if s4['correct']: session_correct['4'] += 1
        
    except Exception as e:
        error_msg = f'Error on QID {qid}: {str(e)}'
        print(f'  ERROR: {error_msg}')
        errors.append(error_msg)
        item_results['error'] = str(e)
    
    elapsed = time.time() - t0
    item_results['time_seconds'] = round(elapsed, 1)
    
    # Stage result summary
    stage_summary = ' | '.join(
        f"S{s}:{'Y' if item_results['stages'].get(s, {}).get('correct', False) else 'N'}" 
        for s in ['1','2','3','4'] if s in item_results.get('stages', {})
    )
    print(f'  {stage_summary} | {elapsed:.1f}s | Expected: {expected_ans}')
    
    # Save to checkpoint after EVERY question
    results[qid] = item_results
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    
    # Append to run log (JSONL for easy tailing)
    log_entry = {'qid': qid, 'time': elapsed, 'stages': {s: item_results['stages'].get(s, {}).get('correct', None) for s in ['1','2','3','4']}}
    with open(RUN_LOG_FILE, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')

session_elapsed = time.time() - session_start
print(f'\n{"="*60}')
print(f'SESSION COMPLETE')
print(f'Questions this session: {session_count}')
print(f'Total completed: {len(results)} / {total_items}')
print(f'Session time: {session_elapsed/60:.1f} minutes')
if errors:
    print(f'Errors: {len(errors)}')
    for e in errors:
        print(f'  - {e}')

---
## 7. Session Summary

In [ ]:
# Compute overall statistics across all completed questions
all_correct = {s: 0 for s in ['1', '2', '3', '4']}
all_parsed = {s: 0 for s in ['1', '2', '3', '4']}
all_searched = 0
all_time = 0
total_done = len(results)

for qid, r in results.items():
    all_time += r.get('time_seconds', 0)
    for s in ['1', '2', '3', '4']:
        stage = r.get('stages', {}).get(s, {})
        if stage.get('correct', False):
            all_correct[s] += 1
        if stage.get('prediction', '') != stage.get('response', ''):
            all_parsed[s] += 1
    if r.get('stages', {}).get('2', {}).get('search_query'):
        all_searched += 1

print('='*60)
print(f'CUMULATIVE RESULTS ({total_done} / {total_items} questions)')
print('='*60)
print(f'\n{"Stage":<25} {"Correct":>8} {"Accuracy":>10} {"Parsed":>10}')
print('-'*55)
for s, name in [('1', 'Baseline'), ('2', '+Tools'), ('3', '+Critique'), ('4', 'Full Pipeline')]:
    acc = all_correct[s] / total_done * 100 if total_done else 0
    parsed_pct = all_parsed[s] / total_done * 100 if total_done else 0
    print(f'Stage {s}: {name:<18} {all_correct[s]:>6}/{total_done}  {acc:>8.1f}%  {parsed_pct:>8.1f}%')

print(f'\nSearch invocations: {all_searched}/{total_done} ({all_searched/total_done*100:.0f}%)' if total_done else '')
print(f'Total compute time: {all_time/3600:.1f} hours')
print(f'Avg per question: {all_time/total_done:.1f}s' if total_done else '')
print(f'Remaining: {total_items - total_done} questions ({max(1, (total_items - total_done) // QUESTIONS_PER_SESSION)} sessions)')

---
## 8. Category Breakdown (Bonus)

In [ ]:
# Category-level accuracy breakdown
from collections import defaultdict

cat_stats = defaultdict(lambda: {'total': 0, 's1': 0, 's2': 0, 's3': 0, 's4': 0})

for item in dataset:
    qid = str(item.get('id', ''))
    if qid not in results:
        continue
    category = item.get('category', item.get('Category', 'Unknown'))
    r = results[qid]
    cat_stats[category]['total'] += 1
    for s in ['1', '2', '3', '4']:
        if r.get('stages', {}).get(s, {}).get('correct', False):
            cat_stats[category][f's{s}'] += 1

print(f'{"Category":<25} {"N":>5} {"S1":>7} {"S2":>7} {"S3":>7} {"S4":>7}')
print('-'*60)
for cat, stats in sorted(cat_stats.items(), key=lambda x: -x[1]['total']):
    n = stats['total']
    print(f'{cat:<25} {n:>5} {stats["s1"]/n*100:>6.1f}% {stats["s2"]/n*100:>6.1f}% {stats["s3"]/n*100:>6.1f}% {stats["s4"]/n*100:>6.1f}%')